#### Colorectal Cancer Microbiome
### Preprocessing and baseline logistic regression

" notebook uses four CRC studies from Meta2DB. The goal is to see if the gut microbiome data can help classify CRC patients and healthy controls."

#### 1. Import libraries and read in data

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

# read in abundance data and sample labels
X = pd.read_csv('data/crc_genus_relative_abundance.csv.gz', index_col=0)
meta = pd.read_csv('data/crc_sample_labels.csv').set_index('iodbID')

# make sure the sample IDs match
X.index = X.index.astype(str)
meta.index = meta.index.astype(str)
X = X.loc[meta.index]
print('Data shape:', X.shape)
print('Missing values:', X.isna().sum().sum())

Data shape: (888, 4139)
Missing values: 0


#### 2. Check the samples

The label is **0 for healthy controls** and **1 for CRC patients**.

In [2]:
print('CRC and control samples:')
print(meta['label'].value_counts())
print('Samples from each study:')
print(pd.crosstab(meta['project_name'], meta['label']))

CRC and control samples:
label
1    469
0    419
Name: count, dtype: int64
Samples from each study:
label              0    1
project_name             
2014_Zeller_CRC   61   90
2015_FengQ        61   46
2015_YuJ          53   75
2019_YachidaS    244  258


#### 3. Split training and testing data

I used Feng, Yu, and Yachida for training. Zeller was kept separate for testing to see how the model performs on another study.

In [3]:
train_rows = meta['project_name'] != '2014_Zeller_CRC'
test_rows = meta['project_name'] == '2014_Zeller_CRC'

XTrain = X.loc[train_rows]
XTest = X.loc[test_rows]
ytrain = meta.loc[train_rows, 'label']
ytest = meta.loc[test_rows, 'label']

print('Training samples:', len(XTrain))
print('Testing samples:', len(XTest))

Training samples: 737
Testing samples: 151


#### 4. Remove rare taxa

Some bacteria only appear in a few samples. I kept taxa detected in at least 5% of the training samples and in at least two training studies. The test data was not used to choose features.

In [4]:
train_studies = meta.loc[train_rows, 'project_name']
prevalence = (XTrain > 0).mean()
study_count = sum(
    (XTrain.loc[train_studies == study] > 0).mean() >= 0.05
    for study in train_studies.unique()
)
keep_taxa = prevalence[(prevalence >= 0.05) & (study_count >= 2)].index
XTrain = XTrain[keep_taxa]
XTest = XTest[keep_taxa]
print('Features after filtering:', XTrain.shape[1])

Features after filtering: 1212


#### 5. CLR transformation

Since the values are relative abundances, I used a centered log-ratio (CLR) transformation. I added a small number first so the zeros would not cause an error when taking the logarithm.

In [5]:
def clr_transform(data):
    log_data = np.log(data + 0.000001)
    return log_data.sub(log_data.mean(axis=1), axis=0)

XTrain = clr_transform(XTrain)
XTest = clr_transform(XTest)
print('CLR transformation finished...')

CLR transformation finished...


#### 6. Scale the data

The scaler is fitted on the training samples and then used on the test samples.

In [6]:
scaler = StandardScaler()
XSTrain = scaler.fit_transform(XTrain)
XSTest = scaler.transform(XTest)
print('Data has been scaled...')

Data has been scaled...


### 7. Training logistic regression model

I started with logistic regression as a simple baseline model before trying more complicated methods.

In [7]:
model = LogisticRegression(l1_ratio=1.0, solver='liblinear', C=0.5,
                           class_weight='balanced', max_iter=1000, random_state=42)
model.fit(XSTrain, ytrain)
print('Model training finished...')

Model training finished...


### 8. Results

Accuracy shows the fraction of correct predictions. ROC AUC measures how well the model separates CRC and control samples at different thresholds.

In [8]:
predY = model.predict(XSTest)
probY = model.predict_proba(XSTest)[:, 1]

print('Test accuracy:', round(accuracy_score(ytest, predY), 3))
print('Test ROC AUC:', round(roc_auc_score(ytest, probY), 3))
print('Confusion matrix:')
print(confusion_matrix(ytest, predY))

Test accuracy: 0.682
Test ROC AUC: 0.742
Confusion matrix:
[[40 21]
 [27 63]]
